# VM SR — Square 보상 정규화: `square_tent12i_rs02` × 3 + `square_fixa015_rs02` × 3 + 대조군 `square_tent12i` × 3 (vm3_new 판형, 0→9 순서)

전용 G4 VM에서 **0부터 9까지 순서대로**. 1번은 런타임 재시작 → 2번부터. 코드 변경 없음(`train.reward_scale`은 9/5 구현, `HANDOFF.md` §17), 9 run(Square 150k, RAM ≈ 160 GB, 5~6시간), 9번이 끝나면 VM 반납. 기준 문서: `HANDOFF.md` §24. 짝 노트북: `colab/vm_square_cap.ipynb`(갈래 B, critic α cap, 코드 변경 있음). 두 VM을 같은 밤에 띄운다.

**배경(§19.8·§19.11·§20.5, `results/2026-09-08/README.md` §6)**: Square에서 목표 엔트로피 12(`square_tent12`)는 42k dip을 없애지만(env 42,032 seed 평균 0.404 vs baseline 0.234) 후반에 무너진다(127k 0.369 < baseline 0.407 < π_dp 0.494). 진단(9/8 `diagnostic_groups.csv`): 엔트로피 12를 지키느라 α가 online 0–5k 0.40 → 10k 0.62 → 20k 1.3 → 50k 6.4 → 110k 20까지 오르고, critic 타깃의 보너스 −α·log π′가 γ = 0.999로 쌓여 `qw_mean`이 143 → 30,916. hard backup(`square_tent12_hq`, β = 0)은 후반을 살리지만(127k 0.490; tent12 s1–3 대비 +0.093 ± 0.020, 3/3) 초반을 잃는다(초기 AUC −0.050 ± 0.040, 42k 0.330). hq에서는 α가 0.17–0.37에 머문다 — 즉 α 폭주는 보너스가 Q를 부풀리는 되먹임의 결과다. Can(γ 0.99, α ≤ 0.5)에서는 같은 것이 약하게만 보인다(§22: Q_W +255까지).

## 갈래 A 질문: Q 스케일을 Can 수준으로 낮추면(`train.reward_scale=0.2`) α가 오를 필요가 없어져 Can 처방이 그대로 전이되는가
- `square_tent12i_rs02_s{1,2,3}`: `train.ent_coef=auto_0.3 train.target_ent=12 train.reward_scale=0.2` — Can의 tent12i 그대로 + 보상 ×0.2(타깃의 r만; 로그의 보상·성공률은 그대로, `qw_mean`·`q_start`는 0.2배 눈금). 예측: α 전 구간 < 1(선형이면 tent12의 1/5 ≈ 3–4, 되먹임이 끊기면 hq처럼 < 0.4), `qw_mean`/0.2 < 500, 42k ≥ 0.404, 127k ≥ 0.472.
- `square_fixa015_rs02_s{1,2,3}`: `train.ent_coef=0.15 train.reward_scale=0.2` — 고정 α, 제어기 없음. "Q 스케일만 맞추면 Can형 고정 α가 그대로 통하나". 주의: Can 5점 스윕의 최선 고정값은 0.3(§19.2)이고 0.15는 Can에서 미검증(`can_prefill_fixa015` 대기 중); 0.15는 §23 해석 4의 보너스 규모(α·H ≈ 1.8/step)를 겨냥한 값이다. 기존 `square_fixalpha_03`(rs 없음)은 5 seed 전부 엔트로피 0 아래로 붕괴, 127k 0.364.
- `square_tent12i_s{1,2,3}` (대조군, 권장): 갈래 A·B의 모든 arm이 tent12i(auto-α **초기 0.3**)인데 Square 비교군은 tent12(초기 1.0)뿐이다. 이 3 run이 있어야 rs02·cap의 이득을 α 초기값 효과와 가를 수 있다. 크레딧이 모자라면 7번 셀의 세 번째 launch 줄을 지운다(6 run, RAM ≈ 105 GB).

## 사전 판정 (정의는 `results/2026-09-08/README.md` §2: online = env − 32,016; 초기 창 online 0–67,984; 최저는 5k 격자 online 5,008·k의 seed 평균곡선; 후반 env 127,136·200 ep)
- **주 지표 1 = env 42,032(online 10,016) seed 평균** — Square dip 시점. 참조: π_dp 0.494, iql 0.503, tent12 0.404(n=5; s1–3 0.407), fixalpha_03 0.358, tent12_hq 0.330, baseline 0.234(s1–3 0.210), mix_prefill 0.030.
- **주 지표 2 = env 127,136 seed 평균**. 참조: mix_prefill 0.622, iql 0.562, π_dp 0.494, tent12_hq 0.490, baseline 0.407(s1–3 0.472), tent12 0.369(s1–3 0.397), fixalpha_03 0.364.
- 보조: 초기 창 평균곡선 최저(tent12 0.326@online 50k — tent12의 최저는 42k가 아니라 후반 하강의 시작점이다; hq 0.287@15k; baseline 0.234@10k), 초기 AUC(tent12 0.404, baseline 0.401, hq 0.378), 102k/127k 평균(env 102,128·127,136; tent12 s1–3 0.367, baseline s1–3 0.455, hq 0.463, mix_prefill 0.624), seed-matched 차이 vs tent12 s1–3 · tent12_hq s1–3 · baseline s1–3 · (있으면) square_tent12i s1–3.
- 진단(판정 아님, 해석 조건): `ent_coef`(rs02에서 α가 1 아래에 머무는지, 고정 0.15 arm은 0.150), `qw_mean`(**rs arm은 0.2배 눈금 → 비교 시 5배**; `q_start − mc_return`도 rs arm에선 눈금이 달라 그대로 못 쓴다), `logp_mean`(≈ −12 유지 여부; fixa015는 붕괴 여부), `mu_absmean`.

| 결과 | 해석 |
|---|---|
| 42k ≥ 0.494 **그리고** 127k ≥ 0.62 | 전이 완성 — "보상 정규화 후 Can 처방"이 두 과제 공통 |
| 42k ≥ 0.404(tent12) 그리고 127k ≥ 0.472(baseline s1–3) | 후반 손실만 해결, 초반은 tent12 동급(π_dp 아래 dip은 남음) |
| 127k ≥ 0.472인데 42k < 0.404 | hq형 — 보너스가 초반에 필요하다는 뜻 |
| 둘 다 아님 | Q 스케일은 원인이 아니라 결과 → 갈래 B(cap) 결과와 대조 |

- 검정력(§21): Square 127k per-seed SD ≈ 0.13 → n=3 SE ≈ 0.075. 127k 판정은 tent12(0.369)와 mix_prefill(0.622)처럼 2 SE 이상 떨어진 참조만 가른다; 0.47 vs 0.49 같은 차이는 n=3으로 못 가른다 → "시사"까지만. 42k는 100 ep(이항 SE 0.05) + seed 이질성.
- 갈래 A와 B의 관계: A가 되면 "스케일 문제였다"가 확정되고 B는 원리적 해법. A가 안 되고 B만 되면 "스케일이 아니라 보너스 누적 자체가 문제". 둘 다 돌려야 갈린다.
- 쓰면 안 되는 문장: 9/8 README §4·§8, §22–23 목록 그대로. 추가로 "reward_scale이 dip 시점을 움직인다"(Can §19.1: ×0.25–×2에서 첫 하락 시점 불변, 기각), "Q_W가 폭주해서 후반이 무너졌다"(인과 아님 — 바로 이 실험이 그 검정), "α 0.15가 답".

## 0. Drive 마운트와 GPU 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Conda 설치 — 실행하면 런타임 자동 재시작

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. 재시작 후 Drive 재마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/dsrl_project'
print(PROJ)

## 3. 최신 o2o 저장소 준비

In [ ]:
%%bash
set -e
git config --global url."https://github.com/".insteadOf "git@github.com:"
if [ -d /content/dsrl/.git ]; then
  git -C /content/dsrl checkout o2o
  git -C /content/dsrl pull --ff-only origin o2o
else
  test ! -e /content/dsrl || { echo '/content/dsrl exists but is not a git checkout'; exit 2; }
  git clone --recurse-submodules -b o2o https://github.com/msp0617/dsrl.git /content/dsrl
fi
git -C /content/dsrl submodule sync --recursive
git -C /content/dsrl submodule update --init --recursive
echo -n 'HEAD: '; git -C /content/dsrl rev-parse --short HEAD

## 4. 캐시에서 conda 환경 복원

In [ ]:
%%bash
set -e
CACHE=/content/drive/MyDrive/dsrl_project/env_cache/dsrl_env.tar.gz
test -s "$CACHE" || { echo "missing $CACHE"; exit 2; }
mkdir -p /usr/local/envs
rm -rf /usr/local/envs/dsrl
tar -xzf "$CACHE" -C /usr/local/envs
source /usr/local/etc/profile.d/conda.sh
conda activate dsrl
python - <<'PY'
import torch, robomimic, robosuite, mujoco, stable_baselines3
assert torch.cuda.is_available(), 'GPU runtime required'
print('torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0))
PY
python /usr/local/envs/dsrl/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py

## 5. Square 정책·정규화 복원과 환경 패치

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
PROJ=/content/drive/MyDrive/dsrl_project
RUNTIME=/content/dsrl/dppo/log
DRIVE=$PROJ/dppo_log
mkdir -p "$RUNTIME"
test -d "$DRIVE" || { echo "missing $DRIVE"; exit 2; }
cp -r "$DRIVE"/. "$RUNTIME"/
CKPT_REL=robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td100_ddim-100steps/2025-04-11_19-13-26_44/checkpoint/state_3000.pt
NORM_REL=robomimic/square/normalization.npz
CKPT_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$CKPT_REL" -print -quit)
NORM_SRC=$(find "$RUNTIME" "$DRIVE" -type f -path "*/$NORM_REL" -print -quit)
test -n "$CKPT_SRC" -a -n "$NORM_SRC" || { echo 'Square policy assets missing'; exit 2; }
mkdir -p "$RUNTIME/$(dirname "$CKPT_REL")" "$RUNTIME/$(dirname "$NORM_REL")"
[ "$CKPT_SRC" = "$RUNTIME/$CKPT_REL" ] || cp -f "$CKPT_SRC" "$RUNTIME/$CKPT_REL"
[ "$NORM_SRC" = "$RUNTIME/$NORM_REL" ] || cp -f "$NORM_SRC" "$RUNTIME/$NORM_REL"
cat > /content/env.sh <<'EOS'
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
export WANDB_MODE=disabled
EOS
python /content/dsrl/colab/patch_env.py
ls -lh "$RUNTIME/$CKPT_REL" "$RUNTIME/$NORM_REL"

## 6. 사전검사 — launch 문자열 테스트, 이번 exp_id 9개가 비어 있는지(있으면 같은 명령이 checkpoint resume), 비교군, RAM·디스크

코드 변경은 없지만 `train.reward_scale`·`train.ent_coef`가 fingerprint에 들어가므로 같은 exp_id의 다른 설정 checkpoint는 resume이 거부된다(그 경우 exp_id를 바꾼다).

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/test_notebook_launches.py 2>&1 | tail -n 2; test ${PIPESTATUS[0]} -eq 0 || { echo 'test_notebook_launches FAILED'; exit 2; }
for E in square_tent12i_rs02_s1 square_tent12i_rs02_s2 square_tent12i_rs02_s3 square_fixa015_rs02_s1 square_fixa015_rs02_s2 square_fixa015_rs02_s3 square_tent12i_s1 square_tent12i_s2 square_tent12i_s3; do
  if [ -f "$PROJ/logs/$E.out" ]; then echo "$E: 이미 있음 -> $(grep '\[done\]\|\[eval\]' $PROJ/logs/$E.out | tail -n 1 | cut -c1-80) (같은 명령이면 resume)"; else echo "$E: 새로 시작"; fi
done
for G in square_tent12 square_tent12_hq square_baseline square_fixalpha_03 square_mix_prefill square_iql; do echo -n "비교군 $G: "; ls -d $PROJ/logs/${G}_s* 2>/dev/null | wc -l; done
echo "processes: $(pgrep -fc '[t]rain_dsrl.py' || true)"; free -g | head -2; df -h /content | tail -n 1

## 7. 9개 시작 — `square_tent12i_rs02` × 3 + `square_fixa015_rs02` × 3 + `square_tent12i` × 3 (150k, 5k 격자). 대조군을 빼려면 세 번째 launch 줄을 지운다

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
source /content/env.sh
cd /content/dsrl
git pull --ff-only origin o2o
PROJ=/content/drive/MyDrive/dsrl_project
mkdir -p "$PROJ/logs"
CFG='--config-path=cfg/robomimic --config-name=dsrl_square.yaml'
COMMON="variant=baseline log_dir=$PROJ/logs train.total_env_steps=150000 offline_mix.mode=none load_offline_data=False"
T12I="train.ent_coef=auto_0.3 train.target_ent=12"
launch () {
  EXP=$1; shift
  if pgrep -af '[t]rain_dsrl.py' | grep -Fq "exp_id=$EXP"; then echo "already running: $EXP"; return; fi
  nohup python train_dsrl.py $CFG exp_id=$EXP "$@" $COMMON > "$PROJ/logs/$EXP.out" 2>&1 &
  echo "started $EXP (pid $!)"
}
for S in 1 2 3; do
  launch square_tent12i_rs02_s$S seed=$S $T12I train.reward_scale=0.2
  launch square_fixa015_rs02_s$S seed=$S train.ent_coef=0.15 train.reward_scale=0.2
  launch square_tent12i_s$S      seed=$S $T12I
done

## 8. 3분 후 자동 확인 — 9개 running, ERR 없음, 인자에 `train.reward_scale=0.2`(rs 6개)·`train.ent_coef=0.15`(fixa015)·`auto_0.3 … target_ent=12`(tent12i). 30분 뒤 다시 돌리면 train_log의 α·logp·qw_mean(rs arm은 0.2배 눈금)도 찍힌다

In [ ]:
%%bash
sleep 180
source /usr/local/etc/profile.d/conda.sh && conda activate dsrl
cd /content/dsrl
PROJ=/content/drive/MyDrive/dsrl_project
python scripts/inspect_runs.py --proj "$PROJ" --only square_tent12i_rs02_s,square_fixa015_rs02_s,square_tent12i_s
for E in square_tent12i_rs02_s1 square_tent12i_rs02_s2 square_tent12i_rs02_s3 square_fixa015_rs02_s1 square_fixa015_rs02_s2 square_fixa015_rs02_s3 square_tent12i_s1 square_tent12i_s2 square_tent12i_s3; do
  echo "== $E: $(grep '\[budget\]\|\[eval\]\|Traceback\|Error' "$PROJ/logs/$E.out" | tail -n 2 | tr '\n' ' ' | cut -c1-160)"
  T=$PROJ/logs/$E/train_log.csv
  [ -f "$T" ] && awk -F, 'NR==1{for(i=1;i<=NF;i++)c[$i]=i} END{printf "   train_log last: env_steps=%s ent_coef=%s critic_ent_coef=%s logp_mean=%s qw_mean=%s mu=%s\n", $c["env_steps"], $c["ent_coef"], $c["critic_ent_coef"], $c["logp_mean"], $c["qw_mean"], $c["mu_absmean"]}' "$T"
done
echo '== processes (인자 확인)'; pgrep -af '[t]rain_dsrl.py' | sed 's/.*exp_id=/exp_id=/' | cut -c1-200 || true
free -g | head -2

## 9. Keepalive — 마지막 프로세스가 끝나면 자동 반납

8번에서 오류가 없을 때만 실행하고 이 셀을 계속 실행 상태로 둡니다.

In [ ]:
import subprocess, time
from pathlib import Path
EXPECTED = [f'square_{kind}_s{s}' for kind in ('tent12i_rs02', 'fixa015_rs02', 'tent12i') for s in (1, 2, 3)]
LOGS = Path('/content/drive/MyDrive/dsrl_project/logs')
assert LOGS.is_dir(), 'Drive가 이 런타임에 마운트돼 있지 않음 — 0번(또는 2번) 셀 먼저'

def running():
    out = subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout
    return [line.strip() for line in out.splitlines() if 'train_dsrl.py' in line and any(f'exp_id={e}' in line for e in EXPECTED)]

def last_event(exp):
    path = LOGS / f'{exp}.out'
    if not path.exists(): return 'NO .out'
    lines = path.read_text(errors='replace').splitlines()[-500:]
    for line in reversed(lines):
        if any(x in line for x in ('[eval]', '[done]', 'Traceback', 'Error')): return line[:100]
    return 'starting'

seen_running = False
while True:
    procs = running()
    events = {e: last_event(e) for e in EXPECTED}
    print(time.strftime('%H:%M'), f'running {len(procs)}/{len(EXPECTED)}', '|', ' | '.join(f'{e}: {v}' for e, v in events.items()), flush=True)
    if procs:
        seen_running = True
    elif seen_running or all(v.startswith('[done]') for v in events.values()):
        print('all VM SR runs stopped -> unassigning', flush=True)
        from google.colab import runtime
        runtime.unassign()
        break
    else:
        print('이 런타임에 실행 중인 run이 없고 [done]도 아님 -> 7번 셀이 안 돌았거나 다른 런타임입니다. 반납하지 않고 종료.', flush=True)
        break
    time.sleep(600)

## 10. 결과 zip (끝난 뒤, CPU 런타임 + 0번 Drive 마운트만으로 됨). 로컬에서는 **새 폴더**(예: `logs/bundle_<날짜>/`)에 풀고 — 옛 번들이 섞인 `~/Downloads/logs`는 쓰지 않는다 —
`python scripts/plot_results.py --logs <새폴더>/logs --out results/<날짜>/square_transfer --axes "square_rs=square_baseline,square_tent12,square_fixalpha_03,square_mix_prefill,square_tent12i,square_tent12i_rs02,square_fixa015_rs02;square_cap=square_baseline,square_tent12,square_tent12_hq,square_tent12i,square_tent12i_cap03,square_tent12i_cap1;can_cap=baseline,tent12i,tent12i_hq,tent12i_cap03"`

rs arm의 `qw_mean`·`q_start`는 0.2배 눈금이다(진단 그림에서 5배로 읽는다).

In [ ]:
%%bash
cd /content/drive/MyDrive/dsrl_project
rm -f csv_bundle.zip
zip -qr csv_bundle.zip logs -i "logs/*/eval_log.csv" "logs/*/train_log.csv" "logs/*.csv" "logs/pretrain/*_log.csv"
ls -lh csv_bundle.zip